# Error Case 7 — 409 Conflict: Entity Already Exists

**Error:** `409 Conflict`  
**Root cause:** Trying to create entity that already exists  
**Fix:** Check existence first OR use GET before POST

## Flow
```
POST /catalogs (name=test-error-catalog)
  → first call: INSERT → 201 Created
  → second call: duplicate key → 409 Conflict
```

In [7]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import sys
import importlib
import polaris_test_utils
importlib.reload(polaris_test_utils)
from polaris_test_utils import *
require_not_prod("error-case test")  # PROD-safe: refuses to run against company PROD

cleanup()
print('✅ Setup complete')

🧹 Cleaning up test resources...
✅ Cleanup done
✅ Setup complete


In [ ]:
# Cell 2: Get valid token first
r = get_token()
print_response(r, 'Valid token response')
valid_token = r.json()['access_token']
print(f'\n✅ Got valid token')

In [8]:
# Cell 2: Create catalog first time — should succeed
r1 = create_catalog()
print_response(r1, 'First create (should be 201)')
assert r1.status_code == 201
print('\n✅ First create succeeded')


First create (should be 201):
  Status:     201
  Request-Id: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000952
  Body: {
  "type": "INTERNAL",
  "name": "test-error-catalog",
  "properties": {
    "polaris.config.drop-with-purge.enabled": "true",
    "default-base-location": "s3a://data-catalog-bucket/test-error-catalog/"
  },
  "createTimestamp": 1780365305539,
  "lastUpdateTimestamp": 0,
  "entityVersion": 1,
  "storageConfigInfo": {
    "endpoint": "http://192.168.139.2:9000",
    "endpointInternal": "http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000",
    "pathStyleAccess": true,
    "storageType": "S3",
    "allowedLocations": [
      "s3a://data-catalog-bucket/test-error-catalog/",
      "s3a://data-catalog-bucket/"
    ]
  }
}

✅ First create succeeded


In [9]:
# Cell 3: Create same catalog again — 409
r2 = create_catalog()
print_response(r2, '409 Duplicate catalog')
request_id = get_request_id(r2)
assert r2.status_code == 409
print(f'\nPolaris-Request-Id: {request_id}')
print('\n✅ 409 confirmed — entity already exists')


409 Duplicate catalog:
  Status:     409
  Request-Id: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000954
  Body: {
  "error": {
    "message": "Cannot create Catalog test-error-catalog. Catalog already exists or resolution failed",
    "type": "AlreadyExistsException",
    "code": 409
  }
}

Polaris-Request-Id: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000954

✅ 409 confirmed — entity already exists


In [10]:
# Cell 4: Also test with principal
r3 = create_principal()
assert r3.status_code == 201
r4 = create_principal()
print_response(r4, '409 Duplicate principal')
assert r4.status_code == 409
print('✅ 409 for duplicate principal too')


409 Duplicate principal:
  Status:     409
  Request-Id: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000958
  Body: {
  "error": {
    "message": "Cannot create Principal test-principal. Principal already exists or resolution failed",
    "type": "AlreadyExistsException",
    "code": 409
  }
}
✅ 409 for duplicate principal too


In [11]:
# Cell 5: Find log in OpenSearch
wait_for_logs(5)
request_id = get_request_id(r4)
hits = search_409_errors(minutes_ago=1)
print_logs(hits)

⏳ Waiting 5s for logs to reach OpenSearch...
Found 10 log entries:

ℹ️  [2026-06-02T01:55:16.458Z] INFO
   logger    : io.quarkus.http.access-log
   requestId : 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000958
   realmId   : POLARIS
   message   : 192.168.194.1 - root [02/Jun/2026:01:55:16 +0000] "POST /api/management/v1/principals HTTP/1.1" 409 152

ℹ️  [2026-06-02T01:55:16.457Z] INFO
   logger    : org.apache.polaris.service.exception.IcebergExceptionMapper
   requestId : 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000958
   realmId   : POLARIS
   message   : Handling runtimeException Cannot create Principal test-principal. Principal already exists or resolution failed

ℹ️  [2026-06-02T01:55:10.206Z] INFO
   logger    : io.quarkus.http.access-log
   requestId : 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000954
   realmId   : POLARIS
   message   : 192.168.194.1 - root [02/Jun/2026:01:55:10 +0000] "POST /api/management/v1/catalogs HTTP/1.1" 409 152

ℹ️  [2026-06-0

In [12]:
# Cell 6: Fix — check existence before create
r_check = requests.get(f'{BASE_MGMT}/catalogs/{TEST_CATALOG}', headers=h())
print(f'Check existence: {r_check.status_code}')
if r_check.status_code == 200:
    print('Catalog already exists — skip create')
else:
    r_create = create_catalog()
    print(f'Created: {r_create.status_code}')

cleanup()
print('\n=== Summary ===')
print('Error:    409 Conflict')
print('Cause:    Duplicate entity (catalog/principal/role/namespace)')
print('Detect:   ERROR message contains already exists/duplicate')
print('Fix:      GET before POST — check existence first')
print('Note:     Grant duplication also causes 409 (known Polaris bug)')

Check existence: 200
Catalog already exists — skip create
🧹 Cleaning up test resources...
✅ Cleanup done

=== Summary ===
Error:    409 Conflict
Cause:    Duplicate entity (catalog/principal/role/namespace)
Detect:   ERROR message contains already exists/duplicate
Fix:      GET before POST — check existence first
Note:     Grant duplication also causes 409 (known Polaris bug)
